# Amaliy mashgʻulot 17. NLP algoritmlarini agglyutinativ morfologiyada stress-test qilish

17-laboratoriyada oʻzbek soʻz shakllari soni qanchalik tez oʻsishini va oddiy algoritmlar apostroflar, kirill yozuvi va tovush almashinuvlarida qanday adashishini koʻrdingiz. Bu yerda hammasini kodda takrorlaymiz:

1. ot va feʼl shakllarining generatorini qoʻshimchalarning toʻgʻri tartibi (morfotaktika) va allomorfiya bilan yozamiz;
2. Heaps qonuni va OOV egri chiziqlari orqali agglyutinativ tilning lugʻati nima uchun deyarli toʻyinmasligini koʻramiz;
3. BPE ni noldan oʻrgatamiz va morfemalarga ajratish bilan solishtiramiz: fertillik va chegaralar mosligi;
4. buzilishlar toʻplamini (apostroflar, kirill yozuvi, qoʻshimchalar zanjiri, k→g, unli tushishi) sodda va barqaror konveyerdan oʻtkazamiz va «oʻtdi / oʻtmadi» jadvalini olamiz.

*«NLP algoritmlari» kursining 17-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Shakllar portlashi: oʻzbek soʻz shakllari generatori

Ot qoʻshimchalarni qatʼiy tartibda qabul qiladi (morfotaktika): oʻzak → son (-lar) → egalik (-im, -ing, -i…) → kelishik (-ning, -ni, -ga, -da, -dan). Oʻrin-payt kelishigi -da dan keyin -gi qoʻshimchasi kelishi mumkin, undan keyin yana son va kelishik: kitob·lar·imiz·da·gi·lar·dan. Oxirida soʻroq yuklamasi -mi turishi mumkin.

Qoʻshimchaning shakli qoʻshnilariga bogʻliq (allomorfiya):
- unlidan keyin bola·m, bola·si, undoshdan keyin kitob·im, kitob·i;
- -ga k va q dan keyin -ka va -qa ga aylanadi: terak·ka, qishloq·qa;
- unli bilan boshlanadigan egalik qoʻshimchasi oldida koʻpgina koʻp boʻgʻinli oʻzaklar oxiridagi k, q jarangli tovushga aylanadi: terak → terag·im, qishloq → qishlogʻ·imiz;
- baʼzi oʻzaklarda oxirgi boʻgʻindagi unli tushib qoladi: ogʻiz → ogʻz·i, burun → burn·i, shahar → shahr·i.

Quyidagi generator shu qoidalarni bajaradi (-dagi ni u ikki morfemaga ajratadi: da·gi; laboratoriyada -dagi kelishik slotining bitta varianti edi). Qavs ichida qoʻshimchalar qoidasiz, shunchaki ulanganda nima chiqishi koʻrsatilgan.

In [ ]:
import re, itertools, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VOWELS = ("a", "e", "i", "o", "u", "oʻ")

# oʻzak -> unli bilan boshlanadigan egalik qoʻshimchasi oldida asos oʻzgarishi:
# "kq": oxirgi k -> g, q -> gʻ (terak -> teragi); "drop": oxirgi boʻgʻin unlisi tushadi (ogʻiz -> ogʻzi)
NOUNS = {
    "kitob": "", "uy": "", "bola": "", "maktab": "", "dala": "", "olma": "", "bogʻ": "", "togʻ": "",
    "qoʻl": "", "doʻst": "", "oʻqituvchi": "", "maʼno": "", "sheʼr": "", "talaba": "", "kutubxona": "",
    "qishloq": "kq", "terak": "kq", "yurak": "kq", "quloq": "kq", "eshik": "kq",
    "ogʻiz": "drop", "burun": "drop", "shahar": "drop", "oʻgʻil": "drop", "qorin": "drop",
}
GLOSS = {}   # oʻzbek guruhi uchun tarjima shart emas
TAGS = {
    "PL": "koʻplik", "P1": "egalik I sh. birl.", "P2": "egalik II sh. birl.", "P3": "egalik III sh.",
    "P1PL": "egalik I sh. koʻp.", "P2PL": "egalik II sh. koʻp.",
    "GEN": "qaratqich k.", "ACC": "tushum k.", "DAT": "joʻnalish k.", "LOC": "oʻrin-payt k.", "ABL": "chiqish k.",
    "ATTR": "-gi (nisbat)", "Q": "soʻroq -mi",
    "NEG": "inkor", "PAST": "oʻtgan z. -di", "PRES": "hozirgi-kelasi z. -a/-y", "PROG": "hozirgi z. -yap", "PERF": "oʻtgan z. -gan",
    "1SG": "I sh. birl.", "2SG": "II sh. birl.", "3SG": "III sh.", "1PL": "I sh. koʻp.", "2PL": "II sh. koʻp.", "3PL": "III sh. koʻp.",
}

POSS = {"P1": ("im", "m"), "P2": ("ing", "ng"), "P3": ("i", "si"), "P1PL": ("imiz", "miz"), "P2PL": ("ingiz", "ngiz")}
CASE = {"GEN": "ning", "ACC": "ni", "DAT": "ga", "LOC": "da", "ABL": "dan"}

def word(m):
    return "".join(x for x, _ in m)

def changed_stem(root):
    kind = NOUNS.get(root, "")
    if kind == "kq":                      # terak -> terag, qishloq -> qishlogʻ
        return root[:-1] + ("g" if root.endswith("k") else "gʻ")
    if kind == "drop":                    # ogʻiz -> ogʻz, shahar -> shahr
        return root[:-2] + root[-1]
    return root

def noun_form(root, pl=False, poss=None, case=None, dagi=False, pl2=False, case2=None, q=False, rules=True):
    """Ot shakli: (morf, teg) juftliklari roʻyxati. rules=False qoʻshimchalarni allomorfiya qoidalarisiz ulaydi."""
    m = [[root, "ROOT"]]
    def add_case(c):
        suf = CASE[c]
        if rules and c == "DAT" and word(m).endswith("k"):
            suf = "ka"                    # terak + ga -> terakka
        elif rules and c == "DAT" and word(m).endswith("q"):
            suf = "qa"                    # qishloq + ga -> qishloqqa
        m.append([suf, c])
    if pl:
        m.append(["lar", "PL"])
    if poss:
        after_cons, after_vowel = POSS[poss]
        if rules and word(m).endswith(VOWELS):
            m.append([after_vowel, poss])  # bola + m
        else:
            if rules and not pl:
                m[0][0] = changed_stem(root)
            m.append([after_cons, poss])   # kitob + im
    if case:
        add_case(case)
    if dagi:                               # -gi faqat -da dan keyin keladi
        m.append(["gi", "ATTR"])
        if pl2:
            m.append(["lar", "PL"])
        if case2:
            add_case(case2)
    if q:
        m.append(["mi", "Q"])
    return [tuple(x) for x in m]

def show(root, **kw):
    m = noun_form(root, **kw)
    plain = word(noun_form(root, rules=False, **kw))
    gloss = " + ".join([GLOSS.get(root, root)] + [TAGS[t] for _, t in m[1:]])
    note = "" if plain == word(m) else "   (qoidasiz: " + plain + ")"
    print(f"{'·'.join(x for x, _ in m):<34} {gloss}{note}")

show("kitob", pl=True, poss="P1PL", case="LOC", dagi=True, pl2=True, case2="ABL")
show("bola", poss="P1")
show("bola", pl=True, poss="P3", case="DAT", q=True)
show("terak", case="DAT")
show("terak", poss="P1", case="GEN")
show("qishloq", case="DAT")
show("qishloq", poss="P1PL", case="DAT")
show("qishloq", pl=True, poss="P1PL", case="DAT")
show("ogʻiz", poss="P3")
show("burun", poss="P3", case="ABL")
show("shahar", poss="P1PL", case="LOC", dagi=True)
show("shahar", pl=True, poss="P1PL", case="LOC")

Bitta oʻzakdan nechta shakl hosil boʻladi? Son (2) × egalik (6, «yoʻq» bilan) × kelishik (6) = 72, xuddi 1-laboratoriyadagi konstruktordagidek; keyin -dagi zanjiri va -mi yuklamasi qoʻshiladi. Feʼlda: boʻlishli/boʻlishsiz × 4 zamon × 6 shaxs-son × -mi. Bu haqiqiy paradigmaning kichik bir qismi: nisbatlar, mayllar, ravishdoshlar, sifatdoshlar va boshqa koʻp shakllar kiritilmagan.

Rus tili bilan solishtiring: «книг-ами» soʻzida bitta qoʻshimcha ham sonni, ham kelishikni bildiradi (fuziya), shuning uchun rus otining oʻnga yaqin shakli bor. Oʻzbek tilida har bir maʼno alohida qoʻshimcha oladi va shakllar soni koʻpayib boradi.

In [ ]:
POSS_OPTS = [None] + list(POSS)
CASE_OPTS = [None] + list(CASE)
CASE2_OPTS = [None, "ACC", "DAT", "ABL"]      # -dagi(-lar) dan keyingi kelishik

def noun_paradigm(root):
    forms = []
    for pl, poss, case in itertools.product([False, True], POSS_OPTS, CASE_OPTS):
        chains = [{}]
        if case == "LOC":
            chains += [dict(dagi=True, pl2=p2, case2=c2) for p2 in (False, True) for c2 in CASE2_OPTS]
        for ch in chains:
            for q in (False, True):
                forms.append(noun_form(root, pl=pl, poss=poss, case=case, q=q, **ch))
    return forms

par = noun_paradigm("kitob")
steps = [("oʻzak", 1), ("+ -lar", 2), ("+ egalik", 2 * 6), ("+ kelishik", 2 * 6 * 6),
         ("+ -dagi (-lar) (+ kelishik)", len(par) // 2), ("+ -mi", len(par))]
for name, n in steps:
    print(f"{name:<32} {n:4d}")
print(f"Otning shakllari kitob: {len(par)} ta shakl, {len({word(m) for m in par})} tasi turlicha\n")

# feʼl: oʻzak (+ inkor -ma) + zamon + shaxs-son (+ -mi)
VERBS = ["yoz", "oʻqi", "ishla", "kel", "chiq"]
TENSE = {"PAST": "di", "PRES": "a", "PROG": "yap", "PERF": "gan"}
PERSON = {
    "PAST": ["m", "ng", "", "k", "ngiz", "lar"],
    "PRES": ["man", "san", "di", "miz", "siz", "dilar"],
    "PROG": ["man", "san", "ti", "miz", "siz", "tilar"],
    "PERF": ["man", "san", "", "miz", "siz", "lar"],
}
PERSON_TAGS = ["1SG", "2SG", "3SG", "1PL", "2PL", "3PL"]

def verb_form(root, tense, person, neg=False, q=False):
    m = [(root, "ROOT")]
    if neg:
        m.append(("ma", "NEG"))
    suf = TENSE[tense]
    if tense == "PRES" and word(m).endswith(VOWELS):
        suf = "y"                          # oʻqi-y-man, yoz-ma-y-man
    elif tense == "PERF" and word(m).endswith("k"):
        suf = "kan"                        # ek-kan
    elif tense == "PERF" and word(m).endswith("q"):
        suf = "qan"                        # chiq-qan
    m.append((suf, tense))
    if PERSON[tense][person]:
        m.append((PERSON[tense][person], PERSON_TAGS[person]))
    if q:
        m.append(("mi", "Q"))
    return m

def verb_paradigm(root):
    return [verb_form(root, t, p, neg, q) for t in TENSE for p in range(6) for neg in (False, True) for q in (False, True)]

for v in ["yoz", "oʻqi", "chiq"]:
    table = pd.DataFrame([[word(verb_form(v, t, p)) for p in range(6)] for t in TENSE],
                         index=[TAGS[t] for t in TENSE], columns=[TAGS[p] for p in PERSON_TAGS])
    print(v + (" — " + GLOSS[v] if v in GLOSS else ""))
    print(table.to_string(), "\n")
print(f"Feʼlning shakllari yoz: {len(verb_paradigm('yoz'))} ta shakl, masalan:", word(verb_form("yoz", "PRES", 0, neg=True, q=True)))

# inventar: soʻz shakli -> (oʻzak, morfemalar); 3- va 4-boʻlimlarda kerak boʻladi
INVENTORY, total = {}, 0
for r in NOUNS:
    for m in noun_paradigm(r):
        INVENTORY[word(m)] = (r, m); total += 1
for v in VERBS:
    for m in verb_paradigm(v):
        INVENTORY[word(m)] = (v, m); total += 1
print(f"Inventar: {len(INVENTORY)} ta soʻz shakli, {len(NOUNS) + len(VERBS)} ta oʻzakdan; turli oʻzaklarda bir xil chiqqan shakllar: {total - len(INVENTORY)}")

plt.figure(figsize=(7, 3))
plt.bar([s for s, _ in steps], [n for _, n in steps], color="#2B55C9")
for i, (_, n) in enumerate(steps):
    plt.text(i, n, str(n), ha="center", va="bottom")
plt.xticks(rotation=20, ha="right")
plt.ylabel("shakllar soni")
plt.title("kitob shakllarining slotlar boʻyicha oʻsishi")
plt.tight_layout()
plt.show()

## 2. Heaps qonuni va OOV: toʻyinmaydigan lugʻat

Heaps qonuni: $N$ ta tokenli matndagi turli soʻz shakllari soni $V$ taxminan $V = K N^{\beta}$, $0 < \beta < 1$ kabi oʻsadi. Tilda bir lemmaga toʻgʻri keladigan shakllar qancha koʻp boʻlsa, $\beta$ shuncha katta: yangi lemmalar deyarli tugagan paytda ham yangi shakllar paydo boʻlaveradi.

Lemmalar taqsimoti bir xil (Zipf qonuni, laboratoriyadagidek 4000 lemma) boʻlgan uchta modellashtirilgan korpusni solishtiramiz: «inglizcha» (lemmaga 4 shakl), «ruscha» (12) va «oʻzbekcha» (336, generatorimizdagidek; laboratoriyada 168 edi, u yerda -dagi dan keyin qoʻshimcha qoʻshilmagan). Lemma ichidagi shakllar ham Zipf boʻyicha taqsimlangan: bir nechta shakl tez-tez, qolganlari kam uchraydi. $\beta$ ni ikki usulda topamiz: log-log shkalada toʻgʻri chiziq (`np.polyfit`) va oddiy shkalada daraja funksiyasini moslashtirish (`scipy.optimize.curve_fit`).

In [ ]:
from scipy.optimize import curve_fit

rng = np.random.default_rng(17)
N_LEMMAS, N_TOKENS, N_TEST = 4000, 200_000, 20_000
FORMS_PER_LEMMA = {"inglizcha": 4, "ruscha": 12, "oʻzbekcha": len(par)}

def zipf(n, s=1.0):
    p = 1.0 / np.arange(1, n + 1) ** s
    return p / p.sum()

corpora = {}
for name, n_forms in FORMS_PER_LEMMA.items():
    lemmas = rng.choice(N_LEMMAS, size=N_TOKENS, p=zipf(N_LEMMAS))
    forms = rng.choice(n_forms, size=N_TOKENS, p=zipf(n_forms))
    corpora[name] = (lemmas, lemmas * 1000 + forms)     # soʻz shakli kodi = lemma * 1000 + shakl raqami

def vocab_growth(tokens):
    first = np.zeros(len(tokens), dtype=bool)
    first[np.unique(tokens, return_index=True)[1]] = True   # soʻz birinchi marta qayerda uchragan
    return np.cumsum(first)                                  # har bir N uchun V(N)

Ns = np.unique(np.logspace(2, np.log10(N_TOKENS - N_TEST), 30).astype(int))
plt.figure(figsize=(6, 4))
for name, (lem, tok) in corpora.items():
    V = vocab_growth(tok[:N_TOKENS - N_TEST])[Ns - 1]
    beta, logK = np.polyfit(np.log(Ns), np.log(V), 1)        # log-log da toʻgʻri chiziq: log V = log K + β log N
    (K2, beta2), _ = curve_fit(lambda n, K, b: K * n ** b, Ns, V, p0=(np.exp(logK), beta))
    print(f"{name:<11} V({Ns[-1]}) = {V[-1]:6d}   polyfit (log-log): β = {beta:.3f}, K = {np.exp(logK):5.2f}   curve_fit: β = {beta2:.3f}, K = {K2:5.2f}")
    plt.loglog(Ns, V, "o-", ms=3, label=f"{name}: β = {beta:.2f}")
plt.xlabel("korpusdagi tokenlar N")
plt.ylabel("turli soʻz shakllari V(N)")
plt.title("Heaps qonuni")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Ikkala usulda ham tillar tartibi bir xil, lekin $\beta$ ning oʻzi farq qiladi: log-log shkalada barcha nuqtalar teng huquqli, oddiy shkalada esa moslashtirishni eng katta $N$ lar belgilaydi. Heaps qonuni taqribiy qonun va $\beta$ baholash usuliga bogʻliq, shuning uchun tillarni bir xil usul bilan solishtirish kerak.

**OOV** (out of vocabulary, lugʻatdan tashqari soʻzlar) — test matnidagi tokenlardan oʻquv korpusida uchramaganlari ulushi. Uni oʻquv korpusining turli hajmlari uchun ikki marta hisoblaymiz: soʻz shakllari boʻyicha va lemmalar boʻyicha (goʻyo bizda xatosiz lemmatizator bor). Uzluksiz va punktir chiziqlarni solishtiring: lemmalar boʻyicha uchala til deyarli farq qilmaydi, butun farqni morfologiya yaratadi.

In [ ]:
train_sizes = [1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000, 180_000]
rows = {}
plt.figure(figsize=(6, 4))
for (name, (lem, tok)), color in zip(corpora.items(), ["#17907E", "#2B55C9", "#C94B2B"]):
    test_tok, test_lem = tok[-N_TEST:], lem[-N_TEST:]
    oov_forms = [np.mean(~np.isin(test_tok, tok[:n])) for n in train_sizes]
    oov_lemmas = [np.mean(~np.isin(test_lem, lem[:n])) for n in train_sizes]
    rows[(name, "shakllar")] = oov_forms
    rows[(name, "lemmalar")] = oov_lemmas
    plt.semilogx(train_sizes, oov_forms, "o-", color=color, label=name)
    plt.semilogx(train_sizes, oov_lemmas, "--", color=color)
print("testdagi OOV, %")
print((pd.DataFrame(rows, index=train_sizes).T * 100).round(1).to_string())
plt.xlabel("oʻquv tokenlari")
plt.ylabel("OOV")
plt.title("OOV ulushi: uzluksiz chiziq — shakllar, punktir — lemmalar")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. BPE va morfemalar: fertillik va chegaralar

BPE (2-mavzu) har qadamda eng koʻp uchraydigan qoʻshni belgilar juftini birlashtiradi. Uni 45 ta oʻzbekcha gapda (30 tasi laboratoriyadan, 15 tasi asos oʻzgarishlari bilan) noldan oʻrgatamiz va uzun shakllarni boʻlaklaymiz. Ikki oʻlchov:
- **fertillik** (fertility) — bir soʻzga oʻrtacha nechta token toʻgʻri kelishi (boʻshliqlar boʻyicha tokenizatsiyada 1, harflar boʻyicha esa soʻz uzunligi);
- **chegaralar mosligi**: BPE chegaralarining qanchasi morfema chegaralariga toʻgʻri keladi (aniqlik) va morfema chegaralarining qanchasi topilgan (toʻliqlik), $F_1 = 2PR/(P+R)$.

Toʻgʻri boʻlaklash 1-boʻlimdagi generatordan olinadi: u har bir morfema qayerda tugashini biladi.

In [ ]:
from collections import Counter

UZ_CORPUS = [
    # 17-laboratoriyadagi 30 ta gap
    "kitoblar stol ustida turibdi", "bu kitobni kutubxonadan oldim", "kitobimiz juda qiziqarli ekan",
    "talabalar kutubxonaga bordilar", "biz uyimizda kitob oʻqiymiz", "uylarimiz maktabga yaqin",
    "maktabimizda katta kutubxona bor", "oʻqituvchimiz talabalarga kitob berdi", "talabalarimiz imtihondan oʻtdilar",
    "doʻstlarim shahardan keldilar", "shahrimizdagi maktablar yangi", "kutubxonadagi kitoblarni oʻqidingizmi",
    "bolalar hovlida oʻynayaptilar", "bolalarimiz maktabdan qaytdilar", "men xat yozdim", "u kitob yozyapti",
    "biz maktabda ishlaymiz", "oʻqituvchilar yigʻilishda gapirdilar", "qishlogʻimizdagi uylar chiroyli",
    "doʻstimning kitobini oʻqidim", "ular kutubxonada ishlayaptilar", "kitoblaringizni stolga qoʻying",
    "talabaning daftari qayerda", "maktablarda imtihonlar boshlandi", "bolalarning daftarlari stolda",
    "uyingizdan maktabgacha uzoqmi", "doʻstlarimizga xat yozdik", "kutubxonamizdagi kitoblar eski",
    "talabalar uylariga qaytdilar", "oʻqituvchimizning kitobi yangi",
    # asos oʻzgaradigan yana 15 ta gap
    "Qishlogʻimizda katta bogʻ bor.", "Bogʻimizdagi teraklar juda baland.", "Oʻgʻlim maktabdan kech qaytdi.",
    "Dehqonlar daladan qaytishdi.", "Biz qishloqqa yozda boramiz.", "Shahrimizda koʻp maktablar bor.",
    "Uyimizning derazasidan togʻlar koʻrinadi.", "Teragimizning barglari sargʻaydi.", "Bolaning ogʻzi ochiq qoldi.",
    "Sovuqdan uning burni qizarib ketdi.", "Bolalarimizga yangi daftarlar oldik.", "Shaharlarimiz tobora chiroyli boʻlmoqda.",
    "Qishloqlarimizdagi maktablar yangilandi.", "Dalalarimizdan paxta terildi.", "Qoʻlimdagi kitobni doʻstimga berdim.",
]

def letters(w):
    return re.findall(r"[og]ʻ|.", w)          # oʻ va gʻ bitta harf

def words_of(s):
    return re.findall(r"[a-zʻʼ]+", s.lower())

word_counts = Counter(w for s in UZ_CORPUS for w in words_of(s))
print(f"{len(UZ_CORPUS)} ta gap, {sum(word_counts.values())} ta soʻz qoʻllanishi, {len(word_counts)} ta turli soʻz\n")

def merge_pair(sym, a, b):
    out, i = [], 0
    while i < len(sym):
        if i + 1 < len(sym) and sym[i] == a and sym[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(sym[i]); i += 1
    return out

def train_bpe(word_counts, max_merges=1000, verbose=12):
    vocab = {tuple(letters(w)) + ("</w>",): c for w, c in word_counts.items()}
    merges = []
    for step in range(max_merges):
        pairs = Counter()
        for sym, c in vocab.items():
            for a, b in zip(sym, sym[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        (a, b), freq = max(pairs.items(), key=lambda t: (t[1], t[0]))   # eng koʻp uchragan juftlik
        if freq < 2:                                                     # bir marta uchragan juftliklar birlashtirilmaydi
            break
        merges.append((a, b))
        vocab = {tuple(merge_pair(list(sym), a, b)): c for sym, c in vocab.items()}
        if step < verbose:
            print(f"birlashtirish {step + 1:2d}: {a} + {b} -> {a + b}   ({freq})")
    return merges

def apply_bpe(w, merges):
    sym = letters(w) + ["</w>"]
    for a, b in merges:
        sym = merge_pair(sym, a, b)
    sym[-1] = sym[-1].replace("</w>", "")
    return [s for s in sym if s]

merges = train_bpe(word_counts)
print(f"...\n{len(merges)} ta birlashtirish oʻrganildi; oxirgilari:", ", ".join(a + b for a, b in merges[-8:]))

LONG_WORDS = ["kitoblarimizdagilardan", "uylarimizda", "oʻqituvchilarimizga", "maktablarimizdagi", "talabalaringizning",
              "kutubxonalarimizdan", "qishlogʻimizga", "teragimizning", "shahrimizdagilar", "ogʻzingizdan", "burnidan",
              "bolalarimizgami"]
print()
for w in LONG_WORDS:
    gold = [x for x, _ in INVENTORY[w][1]]
    print(f"{w:<24} BPE: {' | '.join(apply_bpe(w, merges)):<36} morfemalar: {' | '.join(gold)}")

In [ ]:
def boundaries(pieces):
    out, pos = set(), 0
    for p in pieces[:-1]:
        pos += len(p)
        out.add(pos)
    return out

def evaluate(segment, words):
    n_tok = tp = n_pred = n_gold = 0
    for w in words:
        pieces = segment(w)
        assert "".join(pieces) == w
        pred, gold = boundaries(pieces), boundaries([x for x, _ in INVENTORY[w][1]])
        n_tok += len(pieces); n_pred += len(pred); n_gold += len(gold); tp += len(pred & gold)
    p = tp / n_pred if n_pred else float("nan")
    r = tp / n_gold
    f = 2 * p * r / (p + r) if n_pred and p + r else 0.0
    return n_tok / len(words), p, r, f

long_nouns = sorted(w for w, (r, m) in INVENTORY.items() if r in NOUNS and len(m) >= 3)
sample = random.Random(17).sample(long_nouns, 400)
systems = {
    "boʻshliqlar": lambda w: [w],
    "harflar": letters,
    "BPE-25": lambda w: apply_bpe(w, merges[:25]),
    "BPE-75": lambda w: apply_bpe(w, merges[:75]),
    f"BPE-{len(merges)}": lambda w: apply_bpe(w, merges),
    "morfemalar (etalon)": lambda w: [x for x, _ in INVENTORY[w][1]],
}
res = pd.DataFrame([evaluate(f, sample) for f in systems.values()], index=list(systems),
                   columns=["fertillik", "aniqlik", "toʻliqlik", "F1"])
print(f"{len(sample)} ta tasodifiy ot shakli (≥3 morfema)")
print(res.round(3).to_string(na_rep="—"), "\n")

ks = [0, 5, 10, 25, 50, 75, 100, len(merges)]
curve = np.array([evaluate(lambda w: apply_bpe(w, merges[:k]), sample) for k in ks])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ks, curve[:, 0], "o-", color="#2B55C9")
ax[0].set_xlabel("BPE birlashtirishlari soni"); ax[0].set_ylabel("fertillik"); ax[0].set_title("BPE: birlashtirish koʻpaygan sari tokenlar kamayadi")
for j, (lbl, c) in enumerate([("aniqlik", "#17907E"), ("toʻliqlik", "#C94B2B"), ("F1", "#333333")]):
    ax[1].plot(ks, curve[:, j + 1], "o-", color=c, label=lbl)
ax[1].set_xlabel("BPE birlashtirishlari soni"); ax[1].set_ylim(0, 1.05); ax[1].set_title("BPE chegaralarining morfemalarga mosligi"); ax[1].legend()
plt.tight_layout()
plt.show()

Jadvalda nima koʻrinadi. BPE uzun shakllarni morfemalardan maydaroq boʻlaklaydi (taxminan 7 token, morfemalar esa 5 ta), barcha birlashtirishlardan keyin ham uning taxminan har uchinchi kesishi morfema ichidan oʻtadi. Koʻp uchraydigan qoʻshimchalar zanjiri bitta tokenga birlashadi (larimiz, imizning), korpusda deyarli uchramaydigan oʻzaklar esa harflarga sochilib ketadi (b·u·r·n·i·dan). Birlashtirishlar koʻpaygan sari aniqlik oshadi, toʻliqlik esa kamayadi: BPE morfologiyani emas, belgilar juftlari chastotasini optimallashtiradi.

### Haqiqiy modelda

Quyidagi katak faqat Colab muhitida ishlaydi (internet kerak). U `tokenizers` kutubxonasi yordamida xuddi shu gaplarda BPE ni oʻrgatadi va oʻsha soʻzlarni koʻp tilli XLM-RoBERTa modelining tayyor tokenizatori (`FacebookAI/xlm-roberta-base`, faqat tokenizator fayllari yuklanadi, 10–15 MB) bilan boʻlaklaydi. Uning boʻlaklari morfemalarga mos keladimi, koʻring.

In [ ]:
# Faqat Colab uchun: internet kerak. Faqat XLM-RoBERTa tokenizatori fayllari yuklanadi (10–15 MB, model vaznlarisiz).
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    hf_bpe = Tokenizer(models.BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
    hf_bpe.pre_tokenizer = pre_tokenizers.Whitespace()
    alphabet = {ch for w in word_counts for ch in w}
    trainer = trainers.BpeTrainer(vocab_size=len(alphabet) + len(merges) + 1, min_frequency=2,
                                  special_tokens=["[UNK]"], end_of_word_suffix="</w>", show_progress=False)
    hf_bpe.train_from_iterator([" ".join(words_of(s)) for s in UZ_CORPUS], trainer)

    from transformers import AutoTokenizer
    xlmr = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
    for w in LONG_WORDS:
        print(w)
        print("   tokenizers BPE:", hf_bpe.encode(w).tokens)
        print("   XLM-R         :", xlmr.tokenize(w))
except Exception as e:
    print("Ishlamadi (internet hamda tokenizers va transformers kutubxonalari kerak):", e)

## 4. Stress-test: sodda va barqaror konveyer

Vazifa: soʻz shaklidan lugʻatdagi oʻzakni (lemmani) topish. Qidiruv, tasniflash va tarjima shu qadamdan boshlanadi.

- **Sodda konveyer** soʻzni kichik harflarga oʻtkazadi, lugʻatdan qidiradi, topilmasa roʻyxatdan bitta qoʻshimchani kesib tashlaydi.
- **Barqaror konveyer** toʻrtta tuzatish qoʻshadi: apostroflarni normallashtirish, kirill yozuvini lotinga oʻgirish, qoʻshimchalar zanjirini slotlar boʻyicha tahlil qilish va asos oʻzgarishlari uchun teskari qoidalar (terag → terak, ogʻz → ogʻiz).

Apostroflar haqida: qoidaga koʻra oʻ va gʻ da ʻ (U+02BB), tutuq belgisida (maʼno) esa ʼ (U+02BC) yoziladi. Amalda esa odamlar ', `, ‘, ’ terishadi. Kompyuter uchun bular turli belgilar, standart Unicode normallashtirish (NFKC) ham ularni birlashtirmaydi. Quyida yana bir oqibat koʻrinadi: `\w+` muntazam ifodasi ʻ ni harf deb hisoblaydi, ' ni esa yoʻq, shuning uchun o'qituvchi ikki «soʻz»ga boʻlinib ketadi. Bundan tashqari, oʻzbekcha matnlar kirill yozuvida ham yoziladi.

In [ ]:
import unicodedata

APOSTROPHES = "'`‘’´"            # «notoʻgʻri» apostroflar; toʻgʻri ʻ va ʼ ga tegmaymiz

def norm_apostrophe(w):
    # evristika: o va g dan keyin ʻ (oʻ, gʻ), boshqa joylarda ʼ (tutuq belgisi: maʼno, sheʼr)
    return re.sub("([ogOG])[" + APOSTROPHES + "]|[" + APOSTROPHES + "]",
                  lambda m: m.group(1) + "ʻ" if m.group(1) else "ʼ", w)

CYR = dict(zip("абвгдёжзийклмнопрстуфхцчшъьэюяўқғҳ",
               ["a", "b", "v", "g", "d", "yo", "j", "z", "i", "y", "k", "l", "m", "n", "o", "p", "r", "s", "t", "u",
                "f", "x", "ts", "ch", "sh", "ʼ", "", "e", "yu", "ya", "oʻ", "q", "gʻ", "h"]))

def cyr2lat(w):
    out = []
    for i, ch in enumerate(w):
        if ch == "е":                     # е soʻz boshida va unlidan keyin ye deb oʻqiladi
            out.append("ye" if i == 0 or w[i - 1] in "аеёиоуэюяў" else "e")
        else:
            out.append(CYR.get(ch, ch))
    return "".join(out)

word_re = re.compile(r"\w+")
for v in ["oʻqituvchi", "o'qituvchi", "o`qituvchi", "o‘qituvchi", "o’qituvchi", "ўқитувчи", "maʼno", "ma'no"]:
    ap = [ch for ch in v if ch in APOSTROPHES + "ʻʼ"]
    code = f"U+{ord(ap[0]):04X}" if ap else "—"
    nfkc_same = unicodedata.normalize("NFKC", v) in ("oʻqituvchi", "maʼno")
    fixed = norm_apostrophe(cyr2lat(v.lower()))
    print(f"{v:<11} {code:<7} \\w+ -> {str(word_re.findall(v)):<20} NFKC ok: {str(nfkc_same):<5}  norm.: {fixed}")

In [ ]:
def stem_table(alternations):
    # sirtdagi asos -> oʻzak; almashinuvlar bilan terag -> terak, ogʻz -> ogʻiz qoʻshiladi
    table = {r: r for r in NOUNS}
    if alternations:
        for r in NOUNS:
            table.setdefault(changed_stem(r), r)
    return table

NAIVE_SUFFIXES = sorted({"lar", "im", "ing", "i", "imiz", "ingiz", "m", "ng", "si", "miz", "ngiz",
                         "ning", "ni", "ga", "ka", "qa", "da", "dan", "dagi", "mi"}, key=len, reverse=True)
# slotlar boʻyicha qoʻshimchalar zanjiri: son, egalik, kelishik (yoki -da-gi-lar-kelishik), -mi
SUFFIX_CHAIN = re.compile(r"(lar)?(imiz|ingiz|im|ing|i|miz|ngiz|m|ng|si)?(ning|ni|ga|ka|qa|dan|da(gi(lar)?(ni|ga|dan)?)?)?(mi)?")

def make_pipeline(apostrophe=False, cyrillic=False, chain=False, alternations=False):
    def lemmatize(word_):
        w = word_.lower()
        if cyrillic:
            w = cyr2lat(w)
        if apostrophe:
            w = norm_apostrophe(w)
        stems = stem_table(alternations)
        if chain:
            for cut in range(len(w), 0, -1):          # avval eng uzun asos
                if w[:cut] in stems and SUFFIX_CHAIN.fullmatch(w[cut:]):
                    return stems[w[:cut]]
            return None
        if w in stems:
            return stems[w]
        for s in NAIVE_SUFFIXES:                      # koʻpi bilan bitta qoʻshimcha kesiladi
            if w.endswith(s) and w[:-len(s)] in stems:
                return stems[w[:-len(s)]]
        return None
    return lemmatize

naive = make_pipeline()
robust = make_pipeline(apostrophe=True, cyrillic=True, chain=True, alternations=True)

TESTS = [   # (kirish, kutilgan oʻzak, toifa)
    ("kitobda", "kitob", "toza shakl"), ("uylar", "uy", "toza shakl"), ("bolaga", "bola", "toza shakl"),
    ("maktabni", "maktab", "toza shakl"), ("dalada", "dala", "toza shakl"),
    ("bog'da", "bogʻ", "apostroflar"), ("tog`lar", "togʻ", "apostroflar"), ("qo‘lim", "qoʻl", "apostroflar"),
    ("o’qituvchi", "oʻqituvchi", "apostroflar"), ("ma’nosi", "maʼno", "apostroflar"), ("she'r", "sheʼr", "apostroflar"),
    ("o`g`il", "oʻgʻil", "apostroflar"),
    ("китобда", "kitob", "kirill yozuvi"), ("уйлар", "uy", "kirill yozuvi"), ("болага", "bola", "kirill yozuvi"),
    ("ўқитувчи", "oʻqituvchi", "kirill yozuvi"), ("тоғда", "togʻ", "kirill yozuvi"),
    ("kitoblarimizdagilardan", "kitob", "qoʻshimchalar zanjiri"), ("uylarimizda", "uy", "qoʻshimchalar zanjiri"),
    ("bolalarimizgami", "bola", "qoʻshimchalar zanjiri"), ("maktablarimizdagi", "maktab", "qoʻshimchalar zanjiri"),
    ("dalalarimizdan", "dala", "qoʻshimchalar zanjiri"),
    ("teragim", "terak", "k→g, q→gʻ"), ("qishlogʻi", "qishloq", "k→g, q→gʻ"), ("yuragimiz", "yurak", "k→g, q→gʻ"),
    ("qulogʻing", "quloq", "k→g, q→gʻ"), ("eshigi", "eshik", "k→g, q→gʻ"),
    ("ogʻzi", "ogʻiz", "unli tushishi"), ("burnim", "burun", "unli tushishi"), ("shahri", "shahar", "unli tushishi"),
    ("oʻgʻlim", "oʻgʻil", "unli tushishi"), ("qorni", "qorin", "unli tushishi"),
    ("Шаҳримиздаги", "shahar", "aralash"), ("O'G'LIMIZGA", "oʻgʻil", "aralash"),
    ("qishlog'imizdagilar", "qishloq", "aralash"), ("бурнидан", "burun", "aralash"),
    ("КИТОБЛАРИМИЗДА", "kitob", "aralash"), ("ўғлим", "oʻgʻil", "aralash"),
    ("kompyuterda", None, "nazorat"), ("olmalar", "olma", "nazorat"), ("dalasi", "dala", "nazorat"),
    ("bolam", "bola", "nazorat"),
    ("boglarimizda", "bogʻ", "qiyin holatlar"), ("ogzim", "ogʻiz", "qiyin holatlar"),
    ("kitobcham", "kitob", "qiyin holatlar"), ("bolajonim", "bola", "qiyin holatlar"),
]

def mark(ok):
    return "✓" if ok else "✗"

rows = []
for w, gold, cat in TESTS:
    a, b = naive(w), robust(w)
    rows.append([w, gold or "—", cat, f"{a or '—'} {mark(a == gold)}", f"{b or '—'} {mark(b == gold)}"])
table = pd.DataFrame(rows, columns=["kirish", "kutilgan", "toifa", "sodda", "barqaror"])
print(table.to_string(index=False))
for name, f in [("sodda", naive), ("barqaror", robust)]:
    n_ok = sum(f(w) == gold for w, gold, _ in TESTS)
    print(f"{name:<11} aniqlik {n_ok}/{len(TESTS)} = {n_ok / len(TESTS):.1%}")

Har bir test «kirish → kutilgan oʻzak» juftligi va buzilish toifasidan iborat. Tuzatishlarni birma-bir yoqamiz va toifalar boʻyicha aniqlik qanday oʻsishini koʻramiz. «Nazorat» toifasi tuzatishlar hech narsani buzmaganini tekshiradi, «qiyin holatlar» esa barqaror konveyer ham hamma narsaga qodir emasligini koʻrsatadi: tushirib qoldirilgan apostrof (boglarimizda) hamda uning qoidalarida yoʻq soʻz yasovchi -cha va -jon qoʻshimchalari (kitobcham, bolajonim).

Qoʻlda yozilgan testlarni tizimga moslab qoʻyish oson. Shuning uchun avtomatik testlar qoʻshamiz: 1-boʻlimdagi inventardan barcha ot shakllarini olamiz, ʻ belgisini tasodifiy boshqa «apostrof» bilan almashtiramiz va baʼzan soʻzni bosh harflar bilan yozamiz. Agar barqaror konveyer bunday testlarning hammasidan oʻtsa, bu faqat tahlilchi generator bilan kelishilganini bildiradi: ikkalasi uchun umumiy boʻlgan qoidadagi xatoni bunday test topmaydi. Shuning uchun qoʻlda yozilgan «qiyin holatlar» ham, haqiqiy matnlarda tekshirish ham kerak.

In [ ]:
configs = {
    "sodda": make_pipeline(),
    "+ apostroflar": make_pipeline(apostrophe=True),
    "+ kirill yozuvi": make_pipeline(apostrophe=True, cyrillic=True),
    "+ zanjirlar": make_pipeline(apostrophe=True, cyrillic=True, chain=True),
    "+ almashinuvlar": make_pipeline(apostrophe=True, cyrillic=True, chain=True, alternations=True),
}
cats = list(dict.fromkeys(cat for _, _, cat in TESTS))
abl = pd.DataFrame({name: [np.mean([f(w) == g for w, g, c in TESTS if c == cat]) for cat in cats]
                    for name, f in configs.items()}, index=cats)
abl.loc["aniqlik"] = [np.mean([f(w) == g for w, g, _ in TESTS]) for f in configs.values()]
print((abl * 100).round(0).astype(int).to_string())

plt.figure(figsize=(7, 3))
plt.bar(list(configs), abl.loc["aniqlik"] * 100, color=["#999999"] + ["#2B55C9"] * 4)
for i, v in enumerate(abl.loc["aniqlik"] * 100):
    plt.text(i, v, f"{v:.0f}%", ha="center", va="bottom")
plt.ylabel("aniqlik, %"); plt.ylim(0, 105)
plt.title("Tuzatishlar qoʻshilgan sari testlar toʻplamidagi aniqlik")
plt.tight_layout()
plt.show()

In [ ]:
rnd = random.Random(17)

def perturb(w):
    # har bir ʻ ni tasodifiy «apostrof» bilan almashtiramiz, baʼzan soʻzni bosh harflar bilan yozamiz
    w = re.sub("ʻ", lambda m: rnd.choice(["'", "`", "‘", "’"]), w)
    return w.upper() if rnd.random() < 0.3 else w

noun_forms = [(w, r) for w, (r, m) in INVENTORY.items() if r in NOUNS]
perturbed = [(perturb(w), r) for w, r in noun_forms]
print(f"Inventardagi barcha ot shakllarida avtotest: {len(noun_forms)}")
for name, f in [("sodda", naive), ("barqaror", robust)]:
    clean_acc = np.mean([f(w) == r for w, r in noun_forms])
    pert_acc = np.mean([f(w) == r for w, r in perturbed])
    print(f"   {name:<11} buzilishsiz: {clean_acc:6.1%}   buzilishlar bilan: {pert_acc:6.1%}")
print("  ", ", ".join(w for w, _ in perturbed[:: len(perturbed) // 8][:8]))

## 5. Topshiriqlar

1. **Yangi testlar: tushirib qoldirilgan apostrof.** Yozishmalarda koʻpincha ogʻzi, bogʻlar, togʻda oʻrniga ogzi, boglar, togda deb yozishadi. Ikkita shunday holat «qiyin holatlar» orasida bor; `TESTS` roʻyxatiga «apostrofsiz» toifasida yana 4 ta qoʻshing va jadvalni qayta hisoblang. Keyin barqaror konveyerga zaxira qadam qoʻshing: soʻz tahlil qilinmasa, har bir o yoki g dan keyin ʻ qoʻyib koʻring. Endi nechta test oʻtadi? «Nazorat» toifasi buzilmadimi?
2. **Yangi oʻzaklar.** `NOUNS` ga oyoq (q→gʻ almashinuvi), oʻrin va singil (unli tushishi) oʻzaklarini qoʻshing. Ularning barcha shakllarini `noun_paradigm` bilan yarating va `robust` har bir shakl uchun toʻgʻri oʻzakni qaytarishini tekshiring. Soʻng oʻz-oʻzini tekshirish katagini ishga tushiring.
3. **Heaps qonuni.** 2-boʻlimni «oʻzbekcha» korpus uchun lemmaga 50 va 2000 ta shakl bilan takrorlang (2000 ta shakl uchun `lemmas * 1000 + forms` soʻz shakli kodini oʻzgartirish kerak boʻladi: nima uchun?). 50 000 ta oʻquv tokenida $\beta$ va OOV qanday oʻzgaradi?
4. **Feʼllarda BPE.** 1-boʻlimdagi feʼl shakllarida (`verb_paradigm`) BPE fertilligi va chegaralar F1 ini hisoblang. BPE qaysi qoʻshimchalarni yaxshiroq oʻrgandi: ot qoʻshimchalarinimi yoki feʼl qoʻshimchalarinimi? Nima uchun? Maslahat: oʻquv gaplarida qaysi shakllar koʻproq uchrashiga qarang.
5. **Kirill yozuvida avtotest.** `lat2cyr` funksiyasini yozing (teskari transliteratsiya: oʻ → ў, gʻ → ғ, sh → ш, ch → ч, q → қ, h → ҳ, x → х, ya → я…) va 4-boʻlimdagi avtotestni barcha ot shakllarining kirillcha variantlarida ishga tushiring. U qanday transliteratsiya xatolarini aniqladi?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# 2-topshiriq uchun oʻz-oʻzini tekshirish: yechgandan keyin ishga tushiring
for w, gold in [("oyogʻimizga", "oyoq"), ("oʻrnimiz", "oʻrin"), ("singlimga", "singil"), ("oyoqlarimdan", "oyoq")]:
    got = robust(w)
    print(mark(got == gold), w, "->", got, "(kutilgan:", gold + ")")